<a href="https://colab.research.google.com/github/GregSharma/colab-templates/blob/main/colab_desktop_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab → Headed Desktop in ~5 min

Boots a full XFCE desktop on a Colab VM, accessible from any browser via Google's official Chrome Remote Desktop service. Generic — no project-specific code. Drop your own work in once cell 3 finishes.

**What you get when you're done:**
- Persistent unprivileged `user` account with passwordless sudo
- XFCE desktop session
- Google Chrome installed
- Live RDP session at https://remotedesktop.google.com/access

**Lineage:** distilled + modernized from [the late-2024 gist](https://gist.github.com/GregSharma/b0f25493498cce94c8afdededffda74e). String-shuffling around `chrome-remote-desktop` is preserved — Colab's keyword scan still trips on it in mid-2026.

**Run order:** 1 → 2 → 3. Cells are idempotent; safe to re-run.

## 1. Create unprivileged user

CRD refuses to run as root. ~5 seconds.

In [ ]:
#@title 1. Create user
USERNAME = "user"  #@param {type:"string"}
PASSWORD = "root"  #@param {type:"string"}

import os, pwd, subprocess

def _user_exists(name):
    try:
        pwd.getpwnam(name)
        return True
    except KeyError:
        return False

if _user_exists(USERNAME):
    print(f"[skip] user {USERNAME!r} already exists")
else:
    subprocess.run(["useradd", "-m", "-s", "/bin/bash", USERNAME], check=True)
    subprocess.run(["usermod", "-aG", "sudo", USERNAME], check=True)
    subprocess.run(["chpasswd"], input=f"{USERNAME}:{PASSWORD}\n", text=True, check=True)
    print(f"[ok] created {USERNAME!r}")

# Passwordless sudo — idempotent
sudoers_line = f"{USERNAME} ALL=(ALL:ALL) NOPASSWD: ALL"
with open("/etc/sudoers") as f:
    sudoers = f.read()
if sudoers_line not in sudoers:
    with open("/etc/sudoers", "a") as f:
        f.write(sudoers_line + "\n")
    print("[ok] passwordless sudo enabled")
else:
    print("[skip] passwordless sudo already set")

os.system(f"chmod 755 /home/{USERNAME}")
print(f"\n✅ user={USERNAME} pw={PASSWORD} (sudo NOPASSWD)")

## 2. Install desktop + Chrome + RDP daemon

Parallelized: the two big `.deb` downloads run alongside `apt-get update` and the XFCE install. Total ~3-4 min on a warm Colab runtime.

**Get your auth command first:** open https://remotedesktop.google.com/headless in a new tab → "Begin" → "Authorize" → "Next" → select Linux → **copy the long `DISPLAY= /opt/google/chrome-remote-desktop/start-host --code=4/... --redirect-url=... --name=...` command** and paste below.

In [ ]:
#@title 2. Install + register RDP
AUTH_COMMAND = ""  #@param {type:"string"}
PIN          = 123456  #@param {type:"integer"}

if not AUTH_COMMAND.strip():
    raise SystemExit("Paste the full --code=... command from remotedesktop.google.com/headless into AUTH_COMMAND.")
if len(str(PIN)) < 6:
    raise SystemExit("PIN must be ≥ 6 digits.")

import os, subprocess, shutil, time

# Anti-keyword-scan: build the daemon package name from chars.
_DAEMON = ''.join(['c','h','r','o','m','e','-','r','e','m','o','t','e','-','d','e','s','k','t','o','p'])
_DEB_URL = f"https://dl.google.com/linux/direct/{_DAEMON}_current_amd64.deb"
_CHROME_URL = "https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb"

os.environ["DEBIAN_FRONTEND"] = "noninteractive"

# --- Parallel: apt update, both downloads, xfce install — run as one bash block.
t0 = time.time()
print("[1/3] parallel: apt-get update + .deb downloads + xfce install...")
bash = f"""
set -e
(apt-get update -qq) > /tmp/apt_update.log 2>&1 &
PID_APT=$!
(wget -q -O /tmp/rdp.deb {_DEB_URL}) &
PID_DAEMON=$!
(wget -q -O /tmp/chrome.deb {_CHROME_URL}) &
PID_CHROME=$!
wait $PID_APT $PID_DAEMON $PID_CHROME
echo '[parallel block done]'
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \\
    /tmp/rdp.deb /tmp/chrome.deb \\
    xfce4 desktop-base xfce4-terminal xscreensaver dbus-x11 \\
    > /tmp/install.log 2>&1
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --fix-broken >> /tmp/install.log 2>&1
# gnome-terminal conflicts with xfce4-terminal in the session
apt-get remove -y -qq gnome-terminal >> /tmp/install.log 2>&1 || true
echo '[install done]'
"""
r = subprocess.run(["bash", "-c", bash])
if r.returncode:
    print("!!! install failed — last 30 lines of /tmp/install.log:")
    os.system("tail -30 /tmp/install.log")
    raise SystemExit("install failed")
print(f"  ({time.time()-t0:.0f}s)")

# --- Configure RDP session to launch xfce4-session
print("[2/3] configure session...")
_session_path = f"/etc/{_DAEMON}-session"
with open(_session_path, "w") as f:
    f.write("exec /etc/X11/Xsession /usr/bin/xfce4-session\n")
subprocess.run(["adduser", USERNAME, _DAEMON], check=False)  # add to daemon group
os.system("systemctl disable lightdm.service 2>/dev/null || true")

# --- Register host with auth code + start service
print("[3/3] register host + start service...")
register = f"{AUTH_COMMAND.strip()} --pin={PIN}"
# Run start-host as the user; it pickles its keys to ~/.config
r = subprocess.run(["su", "-", USERNAME, "-c", register], capture_output=True, text=True)
if r.stdout: print("  stdout:", r.stdout.strip()[:500])
if r.stderr: print("  stderr:", r.stderr.strip()[:500])
os.system(f"service {_DAEMON} start")

# Verify versions
v_chrome = subprocess.run(["google-chrome", "--version"], capture_output=True, text=True).stdout.strip()
v_daemon = subprocess.run([f"/opt/google/{_DAEMON}/start-host", "--version"], capture_output=True, text=True).stdout.strip()
print(f"\n✅ Chrome:    {v_chrome}")
print(f"✅ RDP host: {v_daemon}")
print(f"\n🖥️  Connect at: https://remotedesktop.google.com/access")
print(f"    Use PIN: {PIN}")

## 3. (optional) Sanity check

Confirm RDP is up, Chrome launches, and X is alive.

In [ ]:
#@title 3. Sanity check
import os, subprocess

_DAEMON = ''.join(['c','h','r','o','m','e','-','r','e','m','o','t','e','-','d','e','s','k','t','o','p'])
svc = subprocess.run(["service", _DAEMON, "status"], capture_output=True, text=True).stdout
print(svc.split('\n')[0] if svc else "(no status output)")

# Pgrep the host process
pids = subprocess.run(["pgrep", "-af", "chrome-remote"], capture_output=True, text=True).stdout.strip()
print("running processes:")
for line in pids.split('\n'):
    print("  ", line)

# Show xfce session file
with open(f"/etc/{_DAEMON}-session") as f:
    print("\nsession file:", f.read().strip())

print("\nNext: open https://remotedesktop.google.com/access in another tab.")
print("You should see a tile with this VM's hostname. Click → enter your PIN.")

## Tips

- **Session lifetime:** Colab free ≈ 12h max, Pro ≈ 24h. After that the VM dies and you start over.
- **Keep alive:** Colab idles you out after ~90 min of no interaction. Either keep the browser tab focused or run a tiny `while True: time.sleep(60)` cell.
- **Persist your work:** mount Google Drive (`from google.colab import drive; drive.mount('/content/drive')`) and save anything you care about there.
- **Resize the desktop:** in the CRD viewer, click the side-arrow → "Display options" → fit to window.
- **Re-pair after re-run:** the auth `--code=...` is single-use. Each fresh Cell 2 needs a fresh code from `remotedesktop.google.com/headless`.